<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Responsible AI</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 4c: Debiasing</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <ul>
        <li><a href="#section1">1. Install and import libraries</a></li>
        <li><a href="#section2">2. Set up Bedrock for inference</a></li>
        <li><a href="#section3">3. Debiasing LLM outputs</a>
            <ul>
                <li><a href="#section3-1">3.1. Debiasing LLMs with prompt templates</a></li>
                <li><a href="#section3-2">3.2. Debiasing LLMs with self-critique</a></li>
            </ul>
        </li>
        <li><a href="#section4">4. Quizzes</a></li>
    </ul>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Install and import libraries</h2>
</div>

Let's start by installing all required packages as specified in the `requirements.txt` file and importing several libraries.

In [ ]:
%%capture
!pip3 install -r ../requirements.txt --quiet

In [ ]:
import warnings, sys

warnings.filterwarnings("ignore")

import json
from IPython.display import Markdown

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Set up Bedrock for inference</h2>
</div>

To get started, set up Bedrock and instantiate an active runtime to query LLMs.

In [ ]:
from langchain_aws import ChatBedrockConverse

bedrock_model_id = "amazon.nova-lite-v1:0"

bedrock_llm = ChatBedrockConverse(
    model=bedrock_model_id,
    temperature=0,
    max_tokens=None,
)

Next, use Bedrock for inference to test everything works as expected:

In [ ]:
Markdown(bedrock_llm.invoke("How are you doing today?").content)

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Debiasing LLMs</h2>
</div>

LLMs can exhibit biased behavior due to various reasons. To remove bias inducing content from the prompt, first the bias inducing content should be identified, which is not always trivial.

There are several options to mitigate bias, such as:
- utilizing automated tools/methods that can identify bias inducing content in prompts 
- adding explicit constraints to discourage models from generating biased responses 

Let's first establish a baseline to see if a model will give a biased output using a seemingly harmless request.

In [ ]:
biased_prompt = "Write a short Python program to determine if the person will be a good scientist based on extracurricular activities. Only provide the Python program and no additional information. "

Let's print the output:

In [ ]:
import textwrap

print(textwrap.dedent(bedrock_llm.invoke(biased_prompt).content.lstrip()))

To repeatedly create examples, use a prompt template.

In [ ]:
from langchain.prompts import PromptTemplate

# Example of a bad LLM
prompt_template_biased = PromptTemplate(
    # add the Claude template
    template="""You like to leverage person's details to make decisions.
                Instruction: {instruction} .""",
    input_variables=["instruction"],
)

query = """Write a short Python program to determine if the person will be a good scientist based on the length of their name. Only provide the Python program and no additional information."""

# set up the chain
biased_result = bedrock_llm.invoke(
    prompt_template_biased.format_prompt(instruction=query).to_string()
).content.lstrip()

In [ ]:
print(textwrap.dedent(biased_result))

Here, it was possible to trick the model as generally hair length is correlated with gender.

<!-- Subsection Header -->
<div id="section3-1" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">3.1. Debiasing LLMs with prompt templates</h3>
</div>

Next, let's try to apply a prompt template that specifically instructs the model to not provide a biased response:

In [ ]:
# create a prompt template
prompt_template_unbiased = PromptTemplate(
    template="""Answer the following instruction, and make sure that the response is not biased and does not discriminate against certain groups of people.\ 
        If response is discriminatory or bias say 'I am a responsible AI model and cannot answer the question as its discriminatory and bias in nature.'\
        Instruction: {instruction} .""",
    input_variables=["instruction"],
)

# use the same query as before
query = """Write a short python program to determine if the person will be a good scientist based on extracurricular activities. Only provide the python program and no additional information."""

# apply the prompt format template
unbiased_template_response = bedrock_llm.invoke(
    prompt_template_unbiased.format_prompt(instruction=query).to_string()
).content.lstrip()

print(textwrap.dedent(unbiased_template_response))

The model clearly refuses to provide an output with this particular prompt template. Let's see if it is possible to use higher level principles to prevent biased or illegal outputs. For this, you will use `ConstitutionalChain`.

<!-- Subsection Header -->
<div id="section3-2" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">3.2. Debiasing LLMs with self-critique</h3>
</div>

The `ConstitutionalChain` is a chain that ensures the output of a language model adheres to a predefined set of constitutional principles. By incorporating specific rules and guidelines, the `ConstitutionalChain` filters and modifies the generated content to align with these principles, thus providing more controlled, ethical, and contextually appropriate responses. This mechanism helps maintain the integrity of the output while minimizing the risk of generating content that may violate guidelines, be offensive, or deviate from the desired context.

First, have a look at the available principles (a sample of 2): 

In [ ]:
from langchain.chains.constitutional_ai.base import ConstitutionalChain
from langchain.chains.constitutional_ai.principles import PRINCIPLES
import random

random.sample(sorted(PRINCIPLES.items()), 2)

Next, pick a relevant principle.

In [ ]:
principle = ConstitutionalChain.get_principles(["uo-ethics-1"])
principle

In [ ]:
from langchain.chains.llm import LLMChain

# create a prompt template
prompt_template_unethical = PromptTemplate(
    template="""You are a genius editor and give responses creative responses based on your persona and follow instructions.
                Instruction: {instruction}\
                """,
    input_variables=["instruction"],
)

unethical_qa_chain = LLMChain(llm=bedrock_llm, prompt=prompt_template_unethical)

constitutional_chain = ConstitutionalChain.from_llm(
    chain=unethical_qa_chain,
    constitutional_principles=principle,
    llm=bedrock_llm,
    verbose=True,
)

constitutional_chain.invoke(
    input="Write a fake product review for a Dyson vacuum cleaner for Amazon.com that has a 1 star rating."
)

It is also possible to set up a custom `ConstitutionalPrinciple`:

In [ ]:
from langchain.chains.constitutional_ai.models import ConstitutionalPrinciple

ethical_principle = ConstitutionalPrinciple(
    name="Ethical Principle",
    critique_request="The model should never engage in writing fake product reviews.",
    revision_request="Rewrite the model's output to state the request was illegal.",
)

# use the same chain as before, but different principle
constitutional_chain = ConstitutionalChain.from_llm(
    chain=unethical_qa_chain,
    constitutional_principles=[ethical_principle],
    llm=bedrock_llm,
    verbose=True,
)

constitutional_chain.invoke(
    input="Write a fake product review for a Dyson vacuum cleaner for Amazon.com that has a 1 star rating."
)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Write your own Constitutional Principle</h4>
        <p>Write your own <code>ConstitutionalPrinciple</code> and ask the model to perform something that goes against your principle.</p>
    </div>
</div>

In [ ]:
############## CODE HERE ####################


############## END OF CODE ##################

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">4. Quizzes</h2>
    <p>Well done on completing the lab! Now, it's time for a brief knowledge assessment.</p>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Knowledge Assessment</h4>
        <p>Answer the following questions to test your understanding of embeddings, document loaders and RAG workflows.</p>
    </div>
</div>

In [ ]:
import sys
sys.path.append('..')

from mlu_utils.quiz_questions import lab4c_question1

lab4c_question1.display()

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have learned how to:</p>
    <ul>
        <li>Be mindful of your own biases in your assumptions and opinions, avoid using harmful stereotypes. The prompt should not contain any harmful stereotypes or biases. This could include anything that is racist, sexist, or otherwise discriminatory.</li>
        <li>Use inclusive language. This means using language that does not discriminate against any particular group of people. For example, instead of saying “mankind,” you could say “humanity.”</li>
        <li>Have humans in the loop. Once you have generated a response from an LLM, get feedback from multiple humans when testing the LLM’s responses.</li>
    </ul>

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Additional Resources</h3>
    <ul>
        <li><a href="https://github.com/microsoft/promptbench">Microsoft PromptBench</a></li>
        <li><a href="https://www.promptingguide.ai/techniques">Prompting Guide Techniques</a></li>
        <li><a href="https://github.com/uptrain-ai/uptrain">UpTrain AI</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!